In [ ]:
!pip install -q transformers==4.44.2 accelerate>=1.0 sentence-transformers

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

NOME_MODELLO = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(NOME_MODELLO)
model = AutoModelForCausalLM.from_pretrained(
    NOME_MODELLO,
    torch_dtype=torch.float16,
    device_map="auto",
)

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx

BASE = "/kaggle/input/datasets/angelo01paldino/datasetcora"

# ordine nodi da cora.content
content = pd.read_csv(f"{BASE}/cora.content", sep="\t", header=None)
paper_ids = content.iloc[:, 0].astype(str).values
n_nodi = len(paper_ids)
id_to_index = {pid: i for i, pid in enumerate(paper_ids)}

# archi da cora.cites
cites = pd.read_csv(f"{BASE}/cora.cites", sep="\t", header=None, names=["cited", "citing"], dtype=str)
G = nx.Graph()
G.add_nodes_from(range(n_nodi))
for cited, citing in zip(cites["cited"], cites["citing"]):
    if cited in id_to_index and citing in id_to_index:
        G.add_edge(id_to_index[citing], id_to_index[cited])

# testo + etichette dal CSV combinato, ordinato per id
tape = pd.read_csv(f"{BASE}/combined.csv").sort_values("id").reset_index(drop=True)

testi = []
for i in range(n_nodi):
    t = tape.loc[i, "T"]; a = tape.loc[i, "A"]
    t = "" if pd.isna(t) else str(t).strip()
    a = "" if pd.isna(a) else str(a).strip()
    if t and a: testi.append(f"{t}. {a}")
    elif t: testi.append(t)
    elif a: testi.append(a)
    else: testi.append("")

etichette = tape["label"].values.astype(int)
print(f"Nodi: {n_nodi}, Archi: {G.number_of_edges()}, Classi: {len(set(etichette))}")

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")

testi_per_embedding = [t[:1000] if t else "" for t in testi]
embeddings = embedder.encode(testi_per_embedding, convert_to_numpy=True, show_progress_bar=True, batch_size=64)
print("Shape embeddings:", embeddings.shape)  

In [ ]:
import torch.nn.functional as F

class ModelloPerplexity:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.device = model.device

    @torch.no_grad()
    def logprob_testo(self, testo_nodo, contesto=""):
        if contesto:
            prompt = contesto + "\n\n" + testo_nodo
            n_ctx_tok = self.tokenizer(contesto + "\n\n", return_tensors="pt").input_ids.shape[1]
        else:
            prompt = testo_nodo
            n_ctx_tok = 1

        ids = self.tokenizer(prompt, return_tensors="pt").input_ids.to(self.device)
        logits = self.model(ids).logits

        logprobs = F.log_softmax(logits[0, :-1], dim=-1)
        target = ids[0, 1:]
        tok_logprob = logprobs[torch.arange(target.shape[0]), target]

        nodo_logprob = tok_logprob[n_ctx_tok - 1:]
        somma = nodo_logprob.sum().item()
        n_token = nodo_logprob.shape[0]
        return somma, n_token

    def perplexity_testo(self, testo_nodo, contesto=""):
        somma, n_token = self.logprob_testo(testo_nodo, contesto)
        if n_token == 0:
            return float("inf")
        return float(torch.exp(torch.tensor(-somma / n_token)))

In [ ]:
m = ModelloPerplexity(model, tokenizer)


In [ ]:
import numpy as np
from tqdm import tqdm


class ScorerPerplexity:
    def __init__(self, modello, testi, embeddings, max_nodi_contesto=20, max_char_testo=250):
        self.modello = modello
        self.testi = testi
        self.embeddings = embeddings
        self.max_nodi_contesto = max_nodi_contesto
        self.max_char_testo = max_char_testo

    def _costruisci_contesto(self, nodo, membri_community):
        altri = [m for m in membri_community if m != nodo]
        if not altri:
            return ""
        # centroide semantico della community (esclusi il nodo stesso)
        centroide = self.embeddings[altri].mean(axis=0)
        # scelgo i nodi piu' vicini al centroide (cosine similarity)
        emb_altri = self.embeddings[altri]
        sim = emb_altri @ centroide / (
            np.linalg.norm(emb_altri, axis=1) * np.linalg.norm(centroide) + 1e-8
        )
        k = min(self.max_nodi_contesto, len(altri))
        idx_top = np.argsort(sim)[::-1][:k]
        selezionati = [altri[i] for i in idx_top]
        pezzi = [self.testi[m][:self.max_char_testo] for m in selezionati if self.testi[m]]
        return "\n\n".join(pezzi)

    def perplexity_nodo(self, nodo, membri_community):
        testo = self.testi[nodo][:self.max_char_testo]
        if not testo:
            return None
        contesto = self._costruisci_contesto(nodo, membri_community)
        return self.modello.perplexity_testo(testo, contesto)

    def ppl_partizione(self, partizione, mostra_avanzamento=True, descrizione="PPL(C)"):
        community = {}
        for nodo, c in enumerate(partizione):
            community.setdefault(c, []).append(nodo)

        ppl_per_nodo = {}
        iteratore = range(len(partizione))
        if mostra_avanzamento:
            iteratore = tqdm(iteratore, desc=descrizione, unit="nodo")

        for nodo in iteratore:
            c = partizione[nodo]
            ppl = self.perplexity_nodo(nodo, community[c])
            if ppl is not None:
                ppl_per_nodo[nodo] = ppl

        if not ppl_per_nodo:
            return float("inf"), {}
        media = float(np.mean(list(ppl_per_nodo.values())))
        return media, ppl_per_nodo

In [ ]:
scorer = ScorerPerplexity(m, testi, embeddings)


In [ ]:
import numpy as np
from tqdm import tqdm


class LocalMovingPerplexity:
    def __init__(self, scorer, grafo, partizione_iniziale, max_iter=6, min_delta=1e-3,
                 dir_salvataggio="/kaggle/working"):
        self.scorer = scorer
        self.grafo = grafo
        self.partizione = np.array(partizione_iniziale).copy()
        self.max_iter = max_iter
        self.min_delta = min_delta
        self.dir_salvataggio = dir_salvataggio
        self._cache = {}
        # tengo traccia della partizione migliore vista
        self.migliore_partizione = self.partizione.copy()
        self.migliore_ppl = float("inf")

    def _membri(self, community_id, partizione=None):
        p = self.partizione if partizione is None else partizione
        return [n for n in range(len(p)) if p[n] == community_id]

    def _ppl_nodo_in_community(self, nodo, community_id, membri_override=None):
        chiave = (nodo, community_id)
        if chiave in self._cache:
            return self._cache[chiave]
        membri = membri_override if membri_override is not None else self._membri(community_id)
        ppl = self.scorer.perplexity_nodo(nodo, membri)
        if ppl is None:
            ppl = float("inf")
        self._cache[chiave] = ppl
        return ppl

    def _community_vicine(self, nodo):
        vicine = {self.partizione[v] for v in self.grafo.neighbors(nodo)}
        vicine.add(self.partizione[nodo])
        return vicine

    def _invalida_cache_community(self, community_id):
        chiavi = [k for k in self._cache if k[1] == community_id]
        for k in chiavi:
            del self._cache[k]

    def ottimizza(self):
        storia = []
        for it in range(self.max_iter):
            spostamenti = 0
            ordine = list(range(len(self.partizione)))

            for nodo in tqdm(ordine, desc=f"Iter {it+1}/{self.max_iter}", unit="nodo"):
                com_attuale = self.partizione[nodo]
                candidate = self._community_vicine(nodo)
                if len(candidate) <= 1:
                    continue

                ppl_per_com = {}
                for c in candidate:
                    membri = self._membri(c)
                    if c != com_attuale:
                        membri = membri + [nodo]
                    ppl_per_com[c] = self._ppl_nodo_in_community(nodo, c, membri_override=membri)

                com_migliore = min(ppl_per_com, key=ppl_per_com.get)
                if com_migliore != com_attuale and \
                   ppl_per_com[com_attuale] - ppl_per_com[com_migliore] > self.min_delta:
                    self.partizione[nodo] = com_migliore
                    self._invalida_cache_community(com_attuale)
                    self._invalida_cache_community(com_migliore)
                    spostamenti += 1

            ppl_media, _ = self.scorer.ppl_partizione(
                self.partizione, mostra_avanzamento=True,
                descrizione=f"Calcolo PPL(C) fine iter {it+1}"
            )
            storia.append({"iterazione": it + 1, "spostamenti": spostamenti, "ppl_media": ppl_media})
            print(f"Iter {it+1}: spostamenti={spostamenti}, PPL(C)={ppl_media:.3f}")

            # aggiorno la partizione migliore se questa e' la piu' bassa finora
            if ppl_media < self.migliore_ppl:
                self.migliore_ppl = ppl_media
                self.migliore_partizione = self.partizione.copy()

            # salvo dopo ogni iterazione (protezione da interruzioni)
            np.save(f"{self.dir_salvataggio}/partizione_iter_{it+1}.npy", self.partizione)
            np.save(f"{self.dir_salvataggio}/partizione_migliore.npy", self.migliore_partizione)

            if spostamenti == 0:
                print("Nessuno spostamento: convergenza raggiunta.")
                break

        print(f"\nMigliore PPL(C) = {self.migliore_ppl:.3f}")
        # restituisco la partizione MIGLIORE, non l'ultima
        return self.migliore_partizione, storia

## `RefinementPerplexity` — Fase di refinement in stile Leiden, adattata alla perplessità

Per ogni community verifica se contiene sottogruppi semanticamente distinti e la suddivide. Opera solo entro i confini di ogni community (non sposta nodi verso l'esterno). Criterio conservativo: un nodo si unisce a un sotto-gruppo solo se la perplessità non peggiora oltre `min_delta`.

### `__init__(scorer, grafo, partizione, min_delta=1e-3, dir_salvataggio="/kaggle/working")`

- `scorer`: oggetto che espone `perplexity_nodo`.
- `self.partizione`: copia in `np.array` della partizione in ingresso.
- `min_delta`: guadagno minimo di perplessità richiesto per accettare uno spostamento interno.

### `_mappa_community()`

1. Costruisce il dizionario `{id_community -> lista di nodi}` scorrendo `self.partizione` con `setdefault`.

### `_raffina_community(membri, nuovo_id_base)`

Suddivide una singola community in sotto-community coese, restituendo `{nodo -> nuovo_id_sottogruppo}`.

2. **Caso base**: se `len(membri) <= 2` la community è troppo piccola per essere suddivisa → tutti i nodi ricevono `nuovo_id_base`.
3. **Inizializzazione a singleton**: `sotto = {n: n}` — ogni nodo è rappresentante del proprio sotto-gruppo. `membri_set` serve per test di appartenenza O(1).
4. `membri_sottogruppo(rappr)` restituisce i nodi correntemente assegnati al rappresentante `rappr`.

5. **Passaggio di aggregazione interna** — per ogni nodo:
   - `vicini_interni` = vicini nel grafo che appartengono alla stessa community; se vuoto, si passa oltre.
   - `candidati` = sotto-gruppi dei vicini interni, uniti a quello attuale del nodo; se ce n'è solo uno, si passa oltre.
   - Per ogni candidato `sg` si costruiscono i membri (aggiungendo `nodo` se `sg != sg_attuale`, a simulare lo spostamento) e si calcola `scorer.perplexity_nodo`; un risultato `None` viene mappato a `float("inf")`.
   - `sg_migliore` = sotto-gruppo con perplessità minima.
   - Lo spostamento avviene solo se `sg_migliore != sg_attuale` **e** `ppl[sg_attuale] - ppl[sg_migliore] > min_delta`.

6. **Rinomina**: si ordinano i rappresentanti distinti rimasti e si costruisce `rimappa` verso id univoci consecutivi a partire da `nuovo_id_base`; si restituisce l'assegnazione finale `{nodo -> nuovo id}`.

### `raffina()`

7. Ottiene la mappa community→membri; inizializza `nuova_partizione` come copia, `prossimo_id = 0` e `suddivisioni = 0`.
8. **Per ogni community** (con barra `tqdm`):
   - Chiama `_raffina_community(membri, prossimo_id)` e scrive i nuovi id in `nuova_partizione`.
   - `n_dopo` = numero di sotto-gruppi ottenuti; se `> 1` incrementa `suddivisioni`.
   - Avanza `prossimo_id += n_dopo`, così che gli id assegnati restino globalmente univoci.
9. Sostituisce `self.partizione` con `nuova_partizione` e stampa il riepilogo `community prima -> community dopo (community suddivise)`.
10. **Salvataggio su disco**: `partizione_refined.npy`.
11. **Ritorno**: `self.partizione`.

In [ ]:
import numpy as np
from tqdm import tqdm


class RefinementPerplexity:
    """
    Fase di refinement in stile Leiden, adattata alla perplessita'.
    Per ogni community, verifica se contiene sottogruppi semanticamente distinti
    e la suddivide. Opera SOLO entro i confini di ogni community (non sposta nodi
    verso l'esterno). Criterio conservativo: un nodo si unisce a un sotto-gruppo
    solo se la perplessita' non peggiora oltre min_delta.
    """

    def __init__(self, scorer, grafo, partizione, min_delta=1e-3,
                 dir_salvataggio="/kaggle/working"):
        self.scorer = scorer
        self.grafo = grafo
        self.partizione = np.array(partizione).copy()
        self.min_delta = min_delta
        self.dir_salvataggio = dir_salvataggio

    def _mappa_community(self):
        m = {}
        for nodo, c in enumerate(self.partizione):
            m.setdefault(c, []).append(nodo)
        return m

    def _raffina_community(self, membri, nuovo_id_base):
        """
        Suddivide una singola community (lista di nodi) in sotto-community coese.
        Restituisce un dizionario nodo -> nuovo_id_sottogruppo.
        Parte da singleton interni e aggrega solo i nodi coerenti che sono
        anche adiacenti nel grafo (entro la community).
        """
        if len(membri) <= 2:
            # community troppo piccola per suddividere: resta unita
            return {n: nuovo_id_base for n in membri}

        membri_set = set(membri)
        # ogni nodo parte come sotto-gruppo singleton
        sotto = {n: n for n in membri}  # nodo -> rappresentante del sotto-gruppo

        def membri_sottogruppo(rappr):
            return [n for n in membri if sotto[n] == rappr]

        # un passaggio di aggregazione interna
        for nodo in membri:
            # vicini nel grafo che stanno nella stessa community
            vicini_interni = [v for v in self.grafo.neighbors(nodo) if v in membri_set]
            if not vicini_interni:
                continue

            sg_attuale = sotto[nodo]
            # sotto-gruppi candidati = quelli dei vicini interni
            candidati = {sotto[v] for v in vicini_interni}
            candidati.add(sg_attuale)
            if len(candidati) <= 1:
                continue

            ppl_per_sg = {}
            for sg in candidati:
                membri_sg = membri_sottogruppo(sg)
                if sg != sg_attuale:
                    membri_sg = membri_sg + [nodo]
                ppl = self.scorer.perplexity_nodo(nodo, membri_sg)
                ppl_per_sg[sg] = ppl if ppl is not None else float("inf")

            sg_migliore = min(ppl_per_sg, key=ppl_per_sg.get)
            if sg_migliore != sg_attuale and \
               ppl_per_sg[sg_attuale] - ppl_per_sg[sg_migliore] > self.min_delta:
                sotto[nodo] = sg_migliore

        # rinomino i sotto-gruppi con id univoci a partire da nuovo_id_base
        rappresentanti = sorted(set(sotto.values()))
        rimappa = {r: nuovo_id_base + i for i, r in enumerate(rappresentanti)}
        return {n: rimappa[sotto[n]] for n in membri}

    def raffina(self):
        mappa = self._mappa_community()
        nuova_partizione = np.array(self.partizione).copy()
        prossimo_id = 0
        suddivisioni = 0

        for c, membri in tqdm(mappa.items(), desc="refinement", unit="community"):
            n_prima = 1
            assegnazione = self._raffina_community(membri, prossimo_id)
            for nodo, nuovo_id in assegnazione.items():
                nuova_partizione[nodo] = nuovo_id
            n_dopo = len(set(assegnazione.values()))
            if n_dopo > 1:
                suddivisioni += 1
            prossimo_id += n_dopo

        self.partizione = nuova_partizione
        n_com = len(set(nuova_partizione))
        print(f"Refinement: {len(mappa)} -> {n_com} community ({suddivisioni} community suddivise)")
        np.save(f"{self.dir_salvataggio}/partizione_refined.npy", self.partizione)
        return self.partizione

In [ ]:
import json

# parto dalla partizione migliore del local-moving (gia' in memoria come partizione_finale,
# oppure la carico da file se la sessione e' nuova)
#part_da_raffinare = partizione_finale  # o np.load("/kaggle/working/partizione_migliore.npy")
part_da_raffinare = np.load("/kaggle/input/datasets/angelo01paldino/datasetcora/partizione_migliore.npy")

print(f"Community prima del refinement: {len(set(part_da_raffinare))}")

ref = RefinementPerplexity(scorer, G, part_da_raffinare)
partizione_refined = ref.raffina()

# valuto la perplessita' della partizione raffinata
ppl_refined, _ = scorer.ppl_partizione(partizione_refined, descrizione="PPL(C) refined")
print(f"PPL(C) dopo refinement: {ppl_refined:.3f}")

np.save("/kaggle/working/partizione_refined_finale.npy", partizione_refined)
print("Salvato.")